In [ ]:
%pip install -e ../../../

In [ ]:
import numpy as np
import pandas as pd

pipe_options_df = pd.read_excel('../pipe_options-static_properties.xlsx', index_col='option_id')
COLUMNS = list(pipe_options_df.index)
Y2000 = pd.Timestamp(year=2000, month=1, day=1)
Y2024 = pd.Timestamp(year=2024, month=1, day=1)

from water_futures_battle.economy.dynamic_properties import EconomyDB
from water_futures_battle.economy.services import adjust_for_inflation

econ_db = EconomyDB.load_from_file('../../economy/economy-dynamic_properties.xlsx')
inflation = econ_db[EconomyDB.INFLATION]['NL0000']
inflation = inflation[inflation.index <= Y2024]

COSTS_2024 = np.array([
    44.58,
    138.48,
    268.73,
    430.15,
    619.57,
    511.64,
    758.90,
    1125.65,
    1553.47,
    2582.52
])

costs_df = pd.DataFrame(
    data=[COSTS_2024],
    columns=COLUMNS,
    index=pd.DatetimeIndex([Y2024], name='timestamp')
)

costs_df = adjust_for_inflation(costs_df.loc[Y2024], Y2024, inflation)

## Emission factors

We take the values in 2025 from the Excel that Lydia prepared.

Historically, we can see the overall emissions in Europe here: https://ourworldindata.org/co2-emissions 

In 2000, overall emissions in Europe where approx 26% more than in 2025

In [ ]:
TOTAL_EUROPE_EMISS_BIL_TONCO2 = np.array([6.16, 6.43, 6.12, 5.61, 5.03, 4.88])
HISTORICAL_MULTIPLIERS = (TOTAL_EUROPE_EMISS_BIL_TONCO2/TOTAL_EUROPE_EMISS_BIL_TONCO2[-1]).reshape(-1, 1)

EMISSIONS_2025 = np.array([
    8.98,
    35.64,
    70.5,
    141.99,
    221.67,
    936.54,
    1104.09,
    1409.37,
    1715.99,
    2402.71
]).reshape(1, -1)

emissions = pd.DataFrame(
    data=HISTORICAL_MULTIPLIERS*EMISSIONS_2025,
    columns=COLUMNS,
    index=pd.date_range(start="2000-01-01", end="2025-01-01", freq="5YS", name="timestamp")
)

In [ ]:
from water_futures_battle.core.base_model import DynamicProperties

pipe_options_dps = DynamicProperties('pipe_options-dynamic_properties')
pipe_options_dps['new_pipe-unit_cost'] = costs_df.loc[[Y2000]].copy()
pipe_options_dps['new_pipe-emissions_factor'] = emissions

pipe_options_dps.dump(path='..')